In [27]:
import tkinter as tk
from tkinter import messagebox
import copy
import random

SIZE = 3            # Kích thước bàn cờ 3x3
WIN_LEN = 3          # Cần đủ 3 quân liên tiếp để thắng
HUMAN = 'X'          # Người chơi
AI = 'O'             # Máy
EMPTY = None         # Ô trống


class CaroBoard:
    """
    Lớp quản lý trạng thái bàn cờ caro 3x3.
    Bàn cờ được lưu dưới dạng ma trận 3x3, mỗi ô là HUMAN, AI hoặc EMPTY.
    """

    def __init__(self):
        self.grid = [[EMPTY for _ in range(SIZE)] for _ in range(SIZE)]

    def copy(self):
        """Tạo bản sao bàn cờ (dùng khi duyệt cây trò chơi, không ảnh hưởng bàn gốc)."""
        new_board = CaroBoard()
        new_board.grid = [row[:] for row in self.grid]
        return new_board

    def get_empty_cells(self):
        """Trả về danh sách các ô còn trống dạng (row, col)."""
        return [(r, c) for r in range(SIZE) for c in range(SIZE) if self.grid[r][c] == EMPTY]

    def make_move(self, row, col, player):
        """Đánh quân player vào ô (row, col) nếu ô đó trống."""
        if self.grid[row][col] == EMPTY:
            self.grid[row][col] = player
            return True
        return False

    def undo_move(self, row, col):
        """Hoàn tác nước đi tại ô (row, col) (dùng khi backtrack trong minimax)."""
        self.grid[row][col] = EMPTY

    def is_full(self):
        """Kiểm tra bàn cờ đã đầy chưa (hòa nếu đầy mà không ai thắng)."""
        return len(self.get_empty_cells()) == 0

    def check_winner(self):
        """
        Kiểm tra xem có người chiến thắng hay không.
        Trả về HUMAN, AI, hoặc None nếu chưa ai thắng.
        Quét theo 4 hướng: ngang, dọc, chéo chính, chéo phụ.
        """
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]  # ngang, dọc, chéo \, chéo /

        for r in range(SIZE):
            for c in range(SIZE):
                player = self.grid[r][c]
                if player == EMPTY:
                    continue
                for dr, dc in directions:
                    count = 1
                    rr, cc = r + dr, c + dc
                    while 0 <= rr < SIZE and 0 <= cc < SIZE and self.grid[rr][cc] == player:
                        count += 1
                        rr += dr
                        cc += dc
                    if count >= WIN_LEN:
                        return player
        return None

    def is_terminal(self):
        """Trạng thái kết thúc: có người thắng hoặc bàn cờ đầy (hòa)."""
        return self.check_winner() is not None or self.is_full()

    def get_state_string(self):
        """Biểu diễn bàn cờ dạng text (debug nếu cần)."""
        lines = []
        for row in self.grid:
            lines.append(' '.join(cell if cell else '.' for cell in row))
        return '\n'.join(lines)

In [28]:
def evaluate_terminal(board, depth):
    """
    Đánh giá điểm số tại trạng thái kết thúc (terminal state).
    - AI thắng: điểm dương, ưu tiên thắng nhanh (cộng thêm theo depth còn lại).
    - HUMAN thắng: điểm âm, ưu tiên để máy thua chậm nhất có thể (depth còn lại).
    - Hòa: 0 điểm.
    Cộng/trừ depth giúp AI chọn nước thắng nhanh nhất hoặc trì hoãn thua lâu nhất.
    """
    winner = board.check_winner()
    if winner == AI:
        return 10 + depth
    elif winner == HUMAN:
        return -10 - depth
    else:
        return 0  # Hòa


def minimax(board, depth, is_maximizing):
    """
    Trả về giá trị (điểm số) tốt nhất của trạng thái `board`
    với góc nhìn AI (maximizing) hoặc HUMAN (minimizing).

    depth: số nước đã đi từ gốc tới hiện tại, dùng để tính điểm ưu tiên thắng nhanh.
    is_maximizing: True nếu lượt hiện tại là AI, False nếu là HUMAN.
    """
    winner = board.check_winner()
    if winner is not None or board.is_full():
        return evaluate_terminal(board, depth)

    empty_cells = board.get_empty_cells()

    if is_maximizing:
        # Lượt của AI -> muốn điểm số lớn nhất
        best_value = float('-inf')
        for (r, c) in empty_cells:
            board.make_move(r, c, AI)
            value = minimax(board, depth + 1, False)
            board.undo_move(r, c)
            best_value = max(best_value, value)
        return best_value
    else:
        # Lượt của HUMAN -> giả định người chơi tối ưu, muốn điểm số nhỏ nhất
        best_value = float('inf')
        for (r, c) in empty_cells:
            board.make_move(r, c, HUMAN)
            value = minimax(board, depth + 1, True)
            board.undo_move(r, c)
            best_value = min(best_value, value)
        return best_value


def find_best_move_minimax(board):
    """
    Duyệt tất cả các nước đi hợp lệ của AI, dùng minimax() để đánh giá,
    và trả về nước đi (row, col) cho điểm số cao nhất.
    """
    best_score = float('-inf')
    best_move = None
    for (r, c) in board.get_empty_cells():
        board.make_move(r, c, AI)
        score = minimax(board, 1, False)  # Sau khi AI đi, tới lượt HUMAN (minimizing)
        board.undo_move(r, c)
        if score > best_score:
            best_score = score
            best_move = (r, c)
    return best_move

In [29]:
def alpha_beta(board, depth, alpha, beta, is_maximizing):
    """
    Phiên bản Minimax có cắt tỉa Alpha-Beta.
    alpha: giá trị tốt nhất hiện tại của Maximizer (AI) trên đường đi từ gốc.
    beta : giá trị tốt nhất hiện tại của Minimizer (HUMAN) trên đường đi từ gốc.
    """
    winner = board.check_winner()
    if winner is not None or board.is_full():
        return evaluate_terminal(board, depth)

    empty_cells = board.get_empty_cells()

    if is_maximizing:
        value = float('-inf')
        for (r, c) in empty_cells:
            board.make_move(r, c, AI)
            value = max(value, alpha_beta(board, depth + 1, alpha, beta, False))
            board.undo_move(r, c)
            alpha = max(alpha, value)
            if alpha >= beta:
                break  # Beta cutoff: nhánh còn lại chắc chắn không tốt hơn
        return value
    else:
        value = float('inf')
        for (r, c) in empty_cells:
            board.make_move(r, c, HUMAN)
            value = min(value, alpha_beta(board, depth + 1, alpha, beta, True))
            board.undo_move(r, c)
            beta = min(beta, value)
            if alpha >= beta:
                break  # Alpha cutoff
        return value


def find_best_move_alpha_beta(board):
    """
    Duyệt tất cả nước đi hợp lệ của AI bằng Alpha-Beta Pruning,
    trả về nước đi (row, col) tốt nhất.
    """
    best_score = float('-inf')
    best_move = None
    alpha = float('-inf')
    beta = float('inf')

    for (r, c) in board.get_empty_cells():
        board.make_move(r, c, AI)
        score = alpha_beta(board, 1, alpha, beta, False)
        board.undo_move(r, c)
        if score > best_score:
            best_score = score
            best_move = (r, c)
        alpha = max(alpha, best_score)  # Cập nhật alpha ở mức gốc

    return best_move

In [30]:
def expectimax(board, depth, is_maximizing):
    """
    is_maximizing = True  -> lượt AI (max node): chọn giá trị lớn nhất.
    is_maximizing = False -> lượt HUMAN (chance node): lấy trung bình kỳ vọng
                              trên tất cả nước đi hợp lệ (giả định ngẫu nhiên,
                              xác suất đều nhau).
    """
    winner = board.check_winner()
    if winner is not None or board.is_full():
        return evaluate_terminal(board, depth)

    empty_cells = board.get_empty_cells()

    if is_maximizing:
        # MAX NODE: lượt của AI, vẫn chọn nước đi tốt nhất
        value = float('-inf')
        for (r, c) in empty_cells:
            board.make_move(r, c, AI)
            value = max(value, expectimax(board, depth + 1, False))
            board.undo_move(r, c)
        return value
    else:
        # CHANCE NODE: lượt của HUMAN, giả định đánh ngẫu nhiên đều xác suất
        total = 0.0
        prob = 1.0 / len(empty_cells)  # Xác suất đều cho mỗi nước đi hợp lệ
        for (r, c) in empty_cells:
            board.make_move(r, c, HUMAN)
            total += prob * expectimax(board, depth + 1, True)
            board.undo_move(r, c)
        return total


def find_best_move_expectimax(board):
    """
    Duyệt tất cả nước đi hợp lệ của AI, dùng expectimax() để tính giá trị
    kỳ vọng cho mỗi nước, trả về nước đi (row, col) có kỳ vọng cao nhất.
    """
    best_score = float('-inf')
    best_move = None
    for (r, c) in board.get_empty_cells():
        board.make_move(r, c, AI)
        score = expectimax(board, 1, False)  # Sau khi AI đi -> lượt HUMAN (chance node)
        board.undo_move(r, c)
        if score > best_score:
            best_score = score
            best_move = (r, c)
    return best_move

In [31]:
import time


class AIPlayer:
    """
    Lớp đại diện cho máy (AI), cho phép chọn 1 trong 3 thuật toán:
    'minimax', 'alpha_beta', 'expectimax' để tính nước đi tốt nhất.
    """

    ALGORITHMS = {
        'minimax': find_best_move_minimax,
        'alpha_beta': find_best_move_alpha_beta,
        'expectimax': find_best_move_expectimax,
    }

    def __init__(self, algorithm='alpha_beta'):
        self.set_algorithm(algorithm)

    def set_algorithm(self, algorithm):
        """Đổi thuật toán đang dùng. algorithm phải thuộc ALGORITHMS."""
        if algorithm not in self.ALGORITHMS:
            raise ValueError(f"Thuật toán không hợp lệ: {algorithm}")
        self.algorithm = algorithm

    def get_move(self, board):
        """
        Tính và trả về nước đi (row, col) tốt nhất cho AI trên `board` hiện tại,
        dùng thuật toán đang được chọn (self.algorithm). Luôn duyệt đầy đủ
        cây trò chơi (full depth) theo đúng thuật toán đã chọn, không rút gọn,
        để đảm bảo tính đúng đắn 100% theo lý thuyết.
        Trả về None nếu bàn cờ không còn ô trống.
        """
        empty_cells = board.get_empty_cells()
        if not empty_cells:
            return None

        func = self.ALGORITHMS[self.algorithm]
        start_time = time.time()
        move = func(board)
        elapsed = time.time() - start_time
        print(f"[{self.algorithm}] Chọn nước đi {move} trong {elapsed:.3f} giây.")
        return move

In [32]:
import threading
 
 
class CaroGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Cờ Caro 3x3 - Minimax / Alpha-Beta / Expectimax")
        self.root.resizable(False, False)
 
        self.board = CaroBoard()
        self.ai_player = AIPlayer(algorithm='alpha_beta')
        self.human_turn = True   # Người luôn đi trước (X)
        self.game_over = False
        self.ai_thinking = False
 
        self._build_top_frame()
        self._build_board_frame()
        self._build_bottom_frame()
 
        self.update_status("Lượt của bạn (X). Hãy chọn một ô để bắt đầu.")
 
    # --------------------------------------------------------
    # XÂY DỰNG GIAO DIỆN
    # --------------------------------------------------------
    def _build_top_frame(self):
        """Khung trên: chọn thuật toán AI."""
        top_frame = tk.Frame(self.root, pady=8)
        top_frame.pack()
 
        tk.Label(top_frame, text="Chọn thuật toán AI:", font=("Arial", 11, "bold")).grid(
            row=0, column=0, columnspan=3, pady=(0, 4)
        )
 
        self.algo_var = tk.StringVar(value='alpha_beta')
        algos = [
            ("Minimax", "minimax"),
            ("Alpha-Beta", "alpha_beta"),
            ("Expectimax", "expectimax"),
        ]
        for i, (label, value) in enumerate(algos):
            tk.Radiobutton(
                top_frame, text=label, variable=self.algo_var, value=value,
                font=("Arial", 10), command=self.on_algorithm_change
            ).grid(row=1, column=i, padx=8)
 
    def _build_board_frame(self):
        """Khung giữa: lưới 3x3 các nút bấm là bàn cờ."""
        board_frame = tk.Frame(self.root, padx=10, pady=10)
        board_frame.pack()
 
        self.buttons = [[None for _ in range(SIZE)] for _ in range(SIZE)]
        for r in range(SIZE):
            for c in range(SIZE):
                btn = tk.Button(
                    board_frame, text="", width=6, height=3,
                    font=("Arial", 22, "bold"),
                    command=lambda r=r, c=c: self.on_cell_click(r, c)
                )
                btn.grid(row=r, column=c, padx=3, pady=3)
                self.buttons[r][c] = btn
 
    def _build_bottom_frame(self):
        """Khung dưới: label trạng thái + các nút điều khiển (reset, đầu hàng)."""
        self.status_label = tk.Label(self.root, text="", font=("Arial", 12), pady=8)
        self.status_label.pack()
 
        control_frame = tk.Frame(self.root, pady=8)
        control_frame.pack()
 
        tk.Button(
            control_frame, text="Chơi lại", font=("Arial", 11),
            bg="#4CAF50", fg="white", width=10, command=self.reset_game
        ).grid(row=0, column=0, padx=8)
 
        tk.Button(
            control_frame, text="Đầu hàng", font=("Arial", 11),
            bg="#f44336", fg="white", width=10, command=self.surrender
        ).grid(row=0, column=1, padx=8)
 
    # --------------------------------------------------------
    # XỬ LÝ SỰ KIỆN
    # --------------------------------------------------------
    def on_algorithm_change(self):
        """Khi người dùng đổi thuật toán AI."""
        algo = self.algo_var.get()
        self.ai_player.set_algorithm(algo)
        algo_display = {
            'minimax': 'Minimax',
            'alpha_beta': 'Alpha-Beta',
            'expectimax': 'Expectimax',
        }[algo]
        self.update_status(f"Đã chọn thuật toán: {algo_display}")
 
    def on_cell_click(self, row, col):
        """Người chơi click vào 1 ô trên bàn cờ."""
        if self.game_over or not self.human_turn or self.ai_thinking:
            return  # Không cho click khi: ván đã kết thúc / không phải lượt người / AI đang nghĩ
 
        if not self.board.make_move(row, col, HUMAN):
            return  # Ô đã có quân, bỏ qua
 
        self.refresh_board()
 
        if self.check_game_end():
            return
 
        # Chuyển sang lượt AI
        self.human_turn = False
        self.update_status("Máy đang suy nghĩ...")
        self.start_ai_turn()
 
    def start_ai_turn(self):
        """Chạy AI trong thread riêng để không làm treo giao diện."""
        self.ai_thinking = True
        thread = threading.Thread(target=self._ai_compute_move, daemon=True)
        thread.start()
 
    def _ai_compute_move(self):
        """
        Hàm chạy trong thread riêng: tính nước đi AI trên một BẢN SAO bàn cờ
        (board.copy()) để tránh xung đột nếu người dùng bấm "Đầu hàng"/"Chơi lại"
        đúng lúc AI đang suy nghĩ.
        """
        board_snapshot = self.board.copy()
        move = self.ai_player.get_move(board_snapshot)
        # Đưa kết quả về luồng chính của Tkinter để cập nhật giao diện an toàn
        self.root.after(0, self._apply_ai_move, move)
 
    def _apply_ai_move(self, move):
        """Áp dụng nước đi AI vào bàn cờ thật (chạy trên luồng chính Tkinter)."""
        self.ai_thinking = False
 
        if self.game_over:
            return  # Người chơi đã đầu hàng/reset trong lúc AI suy nghĩ -> bỏ qua
 
        if move is None:
            return
 
        r, c = move
        self.board.make_move(r, c, AI)
        self.refresh_board()
 
        if self.check_game_end():
            return
 
        self.human_turn = True
        self.update_status("Lượt của bạn (X).")
 
    # --------------------------------------------------------
    # CẬP NHẬT GIAO DIỆN / TRẠNG THÁI
    # --------------------------------------------------------
    def refresh_board(self):
        """Vẽ lại toàn bộ bàn cờ theo trạng thái self.board."""
        for r in range(SIZE):
            for c in range(SIZE):
                value = self.board.grid[r][c]
                text = value if value else ""
                color = "#1976D2" if value == HUMAN else "#D32F2F" if value == AI else "black"
                self.buttons[r][c].config(text=text, fg=color)
 
    def check_game_end(self):
        """
        Kiểm tra ván đấu đã kết thúc chưa (thắng/thua/hòa).
        Nếu kết thúc: khóa toàn bộ bàn cờ, hiện thông báo, trả về True.
        """
        winner = self.board.check_winner()
        if winner == HUMAN:
            self.end_game("Chúc mừng! Bạn đã thắng!")
            return True
        elif winner == AI:
            self.end_game("Máy đã thắng. Chúc bạn may mắn lần sau!")
            return True
        elif self.board.is_full():
            self.end_game("Hòa! Bàn cờ đã đầy.")
            return True
        return False
 
    def end_game(self, message):
        """Kết thúc ván đấu: khóa bàn cờ và hiển thị thông báo."""
        self.game_over = True
        self.lock_board()
        self.update_status(message)
        messagebox.showinfo("Kết thúc ván đấu", message)
 
    def lock_board(self):
        """Disable toàn bộ nút trên bàn cờ (khi ván đấu kết thúc)."""
        for r in range(SIZE):
            for c in range(SIZE):
                self.buttons[r][c].config(state="disabled")
 
    def unlock_board(self):
        """Enable lại toàn bộ nút trên bàn cờ (khi chơi lại)."""
        for r in range(SIZE):
            for c in range(SIZE):
                self.buttons[r][c].config(state="normal")
 
    def update_status(self, text):
        self.status_label.config(text=text)
 
    # --------------------------------------------------------
    # CÁC NÚT ĐIỀU KHIỂN
    # --------------------------------------------------------
    def reset_game(self):
        """Bắt đầu lại ván đấu mới, giữ nguyên thuật toán đang chọn."""
        self.board = CaroBoard()
        self.human_turn = True
        self.game_over = False
        self.ai_thinking = False
        self.unlock_board()
        self.refresh_board()
        self.update_status("Ván mới bắt đầu. Lượt của bạn (X).")
 
    def surrender(self):
        """Người chơi đầu hàng -> AI thắng, kết thúc ván đấu ngay lập tức."""
        if self.game_over:
            return
        self.end_game("Bạn đã đầu hàng. Máy thắng!")
